# DEtection TRansformers (DETR)

### Install required packages

### Import libraries and set configurations

In [ ]:
# Import required libraries
from PIL import Image  # Import Image module from PIL package for image manipulation
import requests  # Import requests library for making HTTP requests
import matplotlib.pyplot as plt  # Import pyplot module from matplotlib package for plotting
%config InlineBackend.figure_format = 'retina'  # Configure figure format for better resolution on retina displays

import torch  # Import PyTorch library for deep learning functionalities
from torch import nn  # Import nn module from torch for neural network building blocks
from torchvision.models import resnet50  # Import pre-trained ResNet-50 model from torchvision
import torchvision.transforms as T  # Import transforms module from torchvision for image transformations

# Disable gradient computation for performance
torch.set_grad_enabled(False)  # This disables gradient computation in PyTorch, which is useful during inference for performance improvement

### Define DETRdemo model

In [ ]:
class DETRdemo(nn.Module):
    """
    Demo DETR implementation.

    Demo implementation of DETR in minimal number of lines, with the
    following differences wrt DETR in the paper:
    * learned positional encoding (instead of sine)
    * positional encoding is passed at input (instead of attention)
    * fc bbox predictor (instead of MLP)
    The model achieves ~40 AP on COCO val5k and runs at ~28 FPS on Tesla V100.
    Only batch size 1 supported.
    """

    def __init__(self, num_classes, hidden_dim=256, nheads=8,
                 num_encoder_layers=6, num_decoder_layers=6):
        super().__init__()
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.nheads = nheads
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.backbone = None
        self.conv = None
        self.transformer = None
        self.linear_class = None
        self.linear_bbox = None
        self.query_pos = None
        self.row_embed = None
        self.col_embed = None

    def initialise(self):
        # create ResNet-50 backbone
        self.backbone = resnet50()  # Instantiate ResNet-50 backbone
        del self.backbone.fc  # Remove fully connected layer

        # create conversion layer
        self.conv = nn.Conv2d(2048, self.hidden_dim, 1)  # 1x1 convolution to reduce dimensionality

        # create a default PyTorch transformer
        self.transformer = nn.Transformer(
            self.hidden_dim, self.nheads, self.num_encoder_layers, self.num_decoder_layers)  # Transformer instantiation

        # prediction heads, one extra class for predicting non-empty slots
        # note that in baseline DETR linear_bbox layer is 3-layer MLP
        self.linear_class = nn.Linear(self.hidden_dim, self.num_classes + 1)  # Linear layer for class prediction
        self.linear_bbox = nn.Linear(self.hidden_dim, 4)  # Linear layer for bounding box prediction

        # output positional encodings (object queries)
        self.query_pos = nn.Parameter(torch.rand(100, self.hidden_dim))  # Parameter for positional encoding of object queries

        # spatial positional encodings
        # note that in baseline DETR we use sine positional encodings
        self.row_embed = nn.Parameter(torch.rand(50, self.hidden_dim // 2))  # Parameter for row-wise positional encoding
        self.col_embed = nn.Parameter(torch.rand(50, self.hidden_dim // 2))  # Parameter for column-wise positional encoding

    def forward(self, inputs):
        # propagate inputs through ResNet-50 up to avg-pool layer
        x = self.backbone.conv1(inputs)  # Convolutional layer 1
        x = self.backbone.bn1(x)  # Batch normalization
        x = self.backbone.relu(x)  # ReLU activation
        x = self.backbone.maxpool(x)  # Max pooling

        x = self.backbone.layer1(x)  # Layer 1 of ResNet-50
        x = self.backbone.layer2(x)  # Layer 2 of ResNet-50
        x = self.backbone.layer3(x)  # Layer 3 of ResNet-50
        x = self.backbone.layer4(x)  # Layer 4 of ResNet-50

        # convert from 2048 to 256 feature planes for the transformer
        h = self.conv(x)  # Apply convolution to reduce dimensionality

        # construct positional encodings
        H, W = h.shape[-2:]  # Height and width of the feature map
        pos = torch.cat([
            self.col_embed[:W].unsqueeze(0).repeat(H, 1, 1),
            self.row_embed[:H].unsqueeze(1).repeat(1, W, 1),
        ], dim=-1).flatten(0, 1).unsqueeze(1)  # Concatenate row and column positional encodings

        # propagate through the transformer
        h = self.transformer(pos + 0.1 * h.flatten(2).permute(2, 0, 1),
                             self.query_pos.unsqueeze(1)).transpose(0, 1)  # Transformer forward pass

        # finally project transformer outputs to class labels and bounding boxes
        return {'pred_logits': self.linear_class(h),  # Predicted class logits
                'pred_boxes': self.linear_bbox(h).sigmoid()}  # Predicted bounding boxes, sigmoided

### Load pretrained model

In [ ]:
# Create an instance of DETRdemo with 91 classes
detr = DETRdemo(num_classes=91)

# Initialize the model parameters and components
detr.initialise()

# Load pre-trained weights from the specified URL
state_dict = torch.hub.load_state_dict_from_url(
    url='https://dl.fbaipublicfiles.com/detr/detr_demo-da2a99e9.pth',
    map_location='cpu', check_hash=True)
detr.load_state_dict(state_dict)

# Set the model to evaluation mode
detr.eval();

### Define COCO classes and colors

In [ ]:
# COCO classes and corresponding colors for visualization
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]
COLORS = [
    [0.000, 0.447, 0.741],  # Blue color for 'person'
    [0.850, 0.325, 0.098],  # Red color for 'bicycle'
    [0.929, 0.694, 0.125],  # Yellow color for 'car'
    [0.494, 0.184, 0.556],  # Purple color for 'motorcycle'
    [0.466, 0.674, 0.188],  # Green color for 'airplane'
    [0.301, 0.745, 0.933]   # Light blue color for 'bus'
]

### Define image transformation and bounding box functions

In [ ]:
# Define image preprocessing transformations
resize = T.Resize(800)  # Resize image to 800x800 pixels
tensor = T.ToTensor()   # Convert image to tensor
normalize = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize image using mean and standard deviation

transform = T.Compose([  # Compose the transformations
    resize,    # Resize the image
    tensor,    # Convert the image to tensor
    normalize  # Normalize the image
])

# Define functions for bounding box post-processing
def box_cxcywh_to_xyxy(x):
    # Convert bounding box from (center_x, center_y, width, height) to (x_min, y_min, x_max, y_max) format
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=1)

def rescale_bboxes(out_bbox, size):
    # Rescale bounding boxes according to the image size
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)
    return b

### Object detection function

In [ ]:
def detect(im, model, transform):
    """
    Perform object detection on the input image using the provided model and transformation.

    Args:
        im (PIL.Image): Input image
        model (nn.Module): Object detection model
        transform (torchvision.transforms.Compose): Image transformation

    Returns:
        probas (Tensor): Predicted class probabilities
        bboxes_scaled (Tensor): Scaled bounding boxes
    """
    # Mean-std normalize the input image (batch-size: 1)
    img = transform(im)  # Apply transformation to the image
    img = img.unsqueeze(0)  # Add batch dimension

    # Ensure image size is within the supported range
    assert img.shape[-2] <= 1600 and img.shape[-1] <= 1600, 'demo model only supports images up to 1600 pixels on each side'

    # Propagate through the model
    outputs = model(img)  # Pass the image through the model

    # Keep only predictions with confidence greater than 0.7
    probas = outputs['pred_logits']  # Extract predicted class logits
    probas = probas.softmax(-1)[0, :, :-1]  # Apply softmax and remove background class
    keep = probas.max(-1).values > 0.7  # Filter out predictions with confidence below 0.7

    # Convert boxes from [0; 1] to image scales
    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)  # Rescale predicted bounding boxes
    return probas[keep], bboxes_scaled  # Return filtered probabilities and scaled bounding boxes

### Load image and perform object detection

In [ ]:
# Load an image
im = Image.open('/usr/local/notebooks/trafficlight.jpg')
# Resize the image
im = im.resize((450, 300), Image.Resampling.BILINEAR)
# Perform object detection
scores, boxes = detect(im, detr, transform)

### Plot results

In [ ]:
def plot_results(pil_img, prob, boxes):
    """
    Plot the results of object detection.

    Args:
        pil_img (PIL.Image): Input image
        prob (Tensor): Predicted class probabilities
        boxes (Tensor): Bounding boxes
    """
    plt.figure(figsize=(16,10))  # Set figure size for display
    plt.imshow(pil_img)  # Display the input image
    ax = plt.gca()  # Get current axis

    # Create iterators
    prob_iter = iter(prob)
    boxes_iter = iter(boxes.tolist())
    colors_iter = iter(COLORS * 100)

    # Iterate until one of the iterators is exhausted
    while True:
        try:
            p = next(prob_iter)
            xmin, ymin, xmax, ymax = next(boxes_iter)
            c = next(colors_iter)

            # Add bounding box rectangle to the plot
            ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                       fill=False, color=c, linewidth=3))
            cl = p.argmax()  # Get the index of the class with highest probability
            text = f'{CLASSES[cl]}: {p[cl]:0.2f}'  # Generate text for class label and probability
            ax.text(xmin, ymin, text, fontsize=15,  # Add text to the plot
                    bbox=dict(facecolor='yellow', alpha=0.5))  # Specify text box properties
        except StopIteration:
            break  # Break out of the loop when any iterator is exhausted

    plt.axis('off')  # Turn off axis
    plt.show()  # Show the plot

# Display the results
plot_results(im, scores, boxes)